In [0]:
import subprocess

CONTROL_SOCKET = "/tmp/databricks-ssh"
SSH_HOST = "databricks@pub.worldb.dedyn.io"
SSH_PORT = 8889
SSH_KEY = "/Workspace/Users/rogermm@gmail.com/.ssh/id_ed25519_databricks_free"

ssh_base = (
    f"ssh "
    f"-S {CONTROL_SOCKET} "
    f"-p {SSH_PORT} "
    f"-i {SSH_KEY} "
    f"-o BatchMode=yes "
    f"{SSH_HOST}"
)

tools_host = "172.18.0.8"
#tools_host = "tools"
host_ip = "192.168.210.33"
host_ip = "localhost"

def start_ssh_tunnel() -> None:
    command = (
        f"ssh "
        f"-M "
        f"-S {CONTROL_SOCKET} "
        f"-o ControlPersist=yes "
        f"-o ServerAliveInterval=30 "
        f"-o ServerAliveCountMax=3 "
        f"-o ExitOnForwardFailure=yes "
        f"-o StrictHostKeyChecking=accept-new "
        f"-o UserKnownHostsFile=/tmp/known_hosts "
        f"-o BatchMode=yes "
        f"-p {SSH_PORT} "
        f"-i {SSH_KEY} "
        f"-L 127.0.0.1:9092:kafka-4:9092 "
        f"-L 127.0.0.1:8080:traefik:80 "
        f"-L {host_ip}:8888:{tools_host}:8888 "
        f"-fNT "
        f"{SSH_HOST}"
    )

    subprocess.run(command, shell=True, check=True)

def check_ssh_tunnel() -> bool:
    result = subprocess.run(
        f"{ssh_base} -O check",
        shell=True,
        text=True,
        capture_output=True,
    )

    print(result.stdout or result.stderr)
    return result.returncode == 0


def stop_ssh_tunnel() -> None:
    subprocess.run(
        f"{ssh_base} -O exit",
        shell=True,
        check=False,
    )

In [0]:
stop_ssh_tunnel()

Control socket connect(/tmp/databricks-ssh): No such file or directory


# Script to start a simple HTTP server using socat
- Listening port
```shell
ip a

cat > /tmp/socat-http-response.sh <<'EOF'
#!/bin/sh

printf 'HTTP/1.1 200 OK\r\n'
printf 'Content-Type: text/plain\r\n'
printf 'Content-Length: 4\r\n'
printf 'Connection: close\r\n'
printf '\r\n'
printf 'test'
EOF

chmod +x /tmp/socat-http-response.sh

socat TCP-LISTEN:8888,reuseaddr,fork EXEC:/tmp/socat-http-response.sh
```


 - Curl

```shell
curl -v http://tools:8888/
```

# Local IP

In [0]:
!ip a | grep "inet 192"

    inet 192.168.210.33/32 scope global eth0


# Start tunnel

In [0]:
start_ssh_tunnel()

# Check the SSH process is running

In [0]:
!ps -ef | grep ssh

spark-4+     220       1  0 14:18 ?        00:00:00 ssh -M -S /tmp/databricks-ssh -o ControlPersist=yes -o ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ExitOnForwardFailure=yes -o StrictHostKeyChecking=accept-new -o UserKnownHostsFile=/tmp/known_hosts -o BatchMode=yes -p 8889 -i /Workspace/Users/rogermm@gmail.com/.ssh/id_ed25519_databricks_free -L 127.0.0.1:9092:kafka-4:9092 -L 127.0.0.1:8080:traefik:80 -L localhost:8888:172.18.0.8:8888 -fNT databricks@pub.worldb.dedyn.io
spark-4+     224      77 25 14:18 pts/0    00:00:00 /bin/bash -c ps -ef | grep ssh
spark-4+     254     224  0 14:18 pts/0    00:00:00 grep ssh


# Check tunnel

In [0]:
check_ssh_tunnel()

Master running (pid=176)



True

# TCP ping

In [0]:
!nc -v localhost 9092
!nc -v localhost 8080
!nc -v localhost 8888

localhost [127.0.0.1] 9092 (?) open
localhost [127.0.0.1] 8080 (http-alt) open
localhost [127.0.0.1] 8888 (?) open


# HTTP Get

In [0]:
!curl -H "Host: remote-service-hostname"  http://{host_ip}:8888/

curl: (56) Recv failure: Connection reset by peer


In [0]:
import requests
from IPython.display import HTML, display

def show_url(url: str):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    

    display(HTML(response.text))

In [0]:
show_url(f"http://{host_ip}:8888")

# Close tunnel

In [0]:
stop_ssh_tunnel()

Exit request sent.


In [0]:
!ps -ef | grep ssh